# Raqib Fraud Detection Exploration Notebook (Legacy Baseline)
## AI Engineering Production Standards (SDA-AIE-113)

> **Warning for AI Engineers:** This notebook represents the exploratory, unstructured code received from the Data Science team.
> It contains deliberate architectural defects marked with `# SMELL`.
> Execute cells strictly top-to-bottom to observe execution-order sensitivity, import delays, and tight framework couplings.

In [1]:
import os
import time
import joblib
import numpy as np
import pandas as pd

# SMELL 1 & 2: Loading heavy model artifact at global scope with relative path
print("[*] Initializing module and loading model artifact...")
t0 = time.perf_counter()

MODEL_PATH = "models/fraud_model.joblib"
if not os.path.exists(MODEL_PATH):
    MODEL_PATH = "../models/fraud_model.joblib"

artifact = joblib.load(MODEL_PATH)
model_pipeline = artifact["pipeline"]
model_version = artifact.get("version", "unknown")
load_duration = time.perf_counter() - t0

print(f"[!] Model version '{model_version}' loaded at import-time in {load_duration:.3f}s")

[*] Initializing module and loading model artifact...


[!] Model version 'v3.2.0' loaded at import-time in 0.742s


In [2]:
# SMELL 2: Hardcoded relative file paths
DATA_PATH = "data/transactions_sample.csv"
if not os.path.exists(DATA_PATH):
    DATA_PATH = "../data/transactions_sample.csv"

df_transactions = pd.read_csv(DATA_PATH)
print(f"[*] Loaded {len(df_transactions)} transactions successfully.")
df_transactions.head(3)

[*] Loaded 5000 transactions successfully.


,transaction_id,amount_sar,channel,is_night
0,TXN-2026-00001,163.38,pos,0
1,TXN-2026-00002,76.25,pos,0
2,TXN-2026-00003,195.83,pos,0


In [3]:
# SMELL 3: In-place mutation of global dataframe. Execution order lives in developer memory.
df_transactions["amount_log"] = np.log1p(df_transactions["amount_sar"])
df_transactions["is_night"] = df_transactions["is_night"].astype(int)

print("[✓] Feature transformation applied directly to global dataframe.")
df_transactions[["transaction_id", "amount_sar", "amount_log", "is_night"]].head(3)

[✓] Feature transformation applied directly to global dataframe.


,transaction_id,amount_sar,amount_log,is_night
0,TXN-2026-00001,163.38,5.102181,0
1,TXN-2026-00002,76.25,4.347047,0
2,TXN-2026-00003,195.83,5.282340,0


In [4]:
# SMELL 5: Hardcoded business thresholds inline
BLOCK_THRESHOLD = 0.85
REVIEW_BAND = 0.15

# SMELL 4: Monolithic God Function mixing DataFrame creation, prediction, and business rules
def score_transaction_row(row):
    try:
        # Rebuilding dataframe on every single invocation
        features = pd.DataFrame([{
            "amount_log": float(np.log1p(row["amount_sar"])),
            "is_night": int(row["is_night"])
        }])

        # Direct framework call without abstraction seam
        probabilities = model_pipeline.predict_proba(features)[0]
        fraud_probability = float(probabilities[1])

        # Inline decision policy
        if fraud_probability >= BLOCK_THRESHOLD:
            decision = "block"
        elif fraud_probability >= (BLOCK_THRESHOLD - REVIEW_BAND):
            decision = "review"
        else:
            decision = "allow"

        return decision, fraud_probability
    except Exception:
        # SMELL 6: Silent failure swallowing - masks fatal bugs
        return "allow", 0.0

print("[✓] Monolithic scoring routine defined.")

[✓] Monolithic scoring routine defined.


In [5]:
print("[*] Running unoptimized batch scoring loop across 5,000 transactions...")
t_batch_start = time.perf_counter()

decisions = []
probabilities = []

for _, row in df_transactions.iterrows():
    dec, prob = score_transaction_row(row)
    decisions.append(dec)
    probabilities.append(prob)

df_transactions["decision"] = decisions
df_transactions["probability"] = probabilities
batch_duration = time.perf_counter() - t_batch_start

output_csv = "scored.csv"
df_transactions.to_csv(output_csv, index=False)

summary_counts = df_transactions["decision"].value_counts().to_dict()
print(f"[✓] Scored {len(df_transactions)} transactions in {batch_duration:.2f}s -> {output_csv}")
print(f"[*] Summary Breakdown: {summary_counts}")

[*] Running unoptimized batch scoring loop across 5,000 transactions...


[✓] Scored 5000 transactions in 4.08s -> scored.csv
[*] Summary Breakdown: {'allow': 4746, 'review': 199, 'block': 55}
